# Figure 6: Timeline with 50% and 95% credible intervals by region

This notebook reads the posterior predictive summaries and writes Figure 6 beside the other visualization notebooks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

_here = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in [_here, *_here.parents]
     if (path / "6_metapopulation_model" / "data").is_dir()),
    _here,
)
SIMULATIONS_DIR = REPO_ROOT / "6_metapopulation_model" / "outputs" / "simulations"
FIGURES_DIR = REPO_ROOT / "6_metapopulation_model" / "visualization"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
stats_path = SIMULATIONS_DIR / "posterior_stats.csv"
predictions_path = SIMULATIONS_DIR / "ss2_posterior_predictions.csv"
figure_path = FIGURES_DIR / "Figure_6_timeline_50_95_ci_by_region.png"

for input_path in (stats_path, predictions_path):
    if not input_path.is_file():
        raise FileNotFoundError(f"Input file not found: {input_path}")


In [ ]:
posterior_stats = pd.read_csv(stats_path)
predictions = pd.read_csv(predictions_path)

required_stats = {
    "COMMUNE", "REGION", "observed_T1", "predicted_mean", "ci_lower", "ci_upper"
}
missing_stats = required_stats.difference(posterior_stats.columns)
if missing_stats:
    raise ValueError(f"Missing columns in {stats_path.name}: {sorted(missing_stats)}")

prediction_rows = []
for column in predictions.columns:
    parts = column.split(" | ", maxsplit=1)
    if len(parts) != 2:
        raise ValueError(f"Unexpected prediction column: {column}")
    commune, region = parts
    values = predictions[column].dropna().to_numpy()
    if values.size == 0:
        raise ValueError(f"No posterior predictions found for: {column}")
    q025, q25, median, q75, q975 = np.percentile(
        values, [2.5, 25, 50, 75, 97.5]
    )
    prediction_rows.append({
        "COMMUNE": commune,
        "REGION": region,
        "predicted_median": median,
        "ci_50_lower": q25,
        "ci_50_upper": q75,
        "ci_95_lower": q025,
        "ci_95_upper": q975,
    })

prediction_summary = pd.DataFrame(prediction_rows)
region_names = {
"Daraâ-Tafilalet": "Drâa-Tafilalet",
"Fés-Meknès": "Fès-Meknès",
}
posterior_stats["REGION"] = posterior_stats["REGION"].replace(region_names)
prediction_summary["REGION"] = prediction_summary["REGION"].replace(region_names)
ps_plot = posterior_stats.merge(
    prediction_summary,
    on=["COMMUNE", "REGION"],
    how="left",
    validate="one_to_one",
)
interval_columns = [
    "predicted_median", "ci_50_lower", "ci_50_upper",
    "ci_95_lower", "ci_95_upper",
]
missing_predictions = ps_plot.loc[ps_plot[interval_columns].isna().any(axis=1)]
if not missing_predictions.empty:
    missing_keys = missing_predictions[["COMMUNE", "REGION"]].to_dict("records")
    raise ValueError(f"Missing predictions for commune-region pairs: {missing_keys}")
ps_plot["within_95ci"] = ps_plot["observed_T1"].between(
    ps_plot["ci_95_lower"], ps_plot["ci_95_upper"]
)
print(f"Loaded {len(ps_plot)} communes across {ps_plot['REGION'].nunique()} regions.")


In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

regions = sorted(ps_plot["REGION"].unique())
n_regions = len(regions)
if n_regions <= 3:
    n_cols, n_rows = n_regions, 1
elif n_regions <= 9:
    n_cols, n_rows = 3, (n_regions + 2) // 3
else:
    n_cols, n_rows = 4, (n_regions + 3) // 4

C_LINE = "#0886da"
C_FILL_50 = "#8bcaf5"
C_FILL_95 = "#d6edfd"
C_OBS = "#fa0202"
seed_region = "Fès-Meknès"
seed_communes = {"ain chkef", "ain chock", "ain chok", "ain chqef"}
y_min = min(ps_plot["observed_T1"].min(), ps_plot["ci_95_lower"].min()) * 0.90
y_max = max(ps_plot["observed_T1"].max(), ps_plot["ci_95_upper"].max()) * 1.15

fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(5.5 * n_cols, 4 * n_rows),
    dpi=300, facecolor="white", sharey=True, squeeze=False,
)
axes = axes.ravel()

for index, region in enumerate(regions):
    ax = axes[index]
    data = ps_plot.loc[ps_plot["REGION"] == region].sort_values("observed_T1")
    x = np.arange(len(data))
    observed = data["observed_T1"].to_numpy()
    predicted = data["predicted_median"].to_numpy()
    commune_names = data["COMMUNE"].to_numpy()
    is_seed = (
        (region == seed_region)
        & data["COMMUNE"].str.lower().str.strip().isin(seed_communes).to_numpy()
    )

    ax.fill_between(x, data["ci_95_lower"], data["ci_95_upper"], color=C_FILL_95, zorder=1)
    ax.fill_between(x, data["ci_50_lower"], data["ci_50_upper"], color=C_FILL_50, zorder=2)
    ax.plot(x, predicted, color=C_LINE, linewidth=2.5, marker="o", markersize=5, zorder=4)
    for point, color in zip(x, np.where(is_seed, C_LINE, C_OBS)):
        ax.scatter(point, observed[point], color=color, s=25, zorder=6)
        ax.plot(
            [point, point], [predicted[point], observed[point]],
            color=color, linewidth=1.5, linestyle=":", alpha=0.5, zorder=3,
        )

    rotation = 90 if len(data) > 15 else 45
    fontsize = 9 if len(data) > 15 else 10 if len(data) > 10 else 11
    ax.set_xticks(x)
    ax.set_xticklabels(
        commune_names, rotation=rotation, ha="right", fontsize=fontsize,
        color="black", fontweight="bold",
    )
    ax.set_ylim(y_min, y_max)
    ax.set_title(region, fontsize=12, fontweight="bold", pad=10)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.5, color="gray")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1.5)

for ax in axes[n_regions:]:
    ax.axis("off")

fig.supxlabel("Commune", fontsize=14, fontweight="bold")
fig.supylabel("Time (ISO weeks)", fontsize=14, fontweight="bold")
legend_elements = [
    mpatches.Patch(facecolor=C_FILL_95, label="95% Credible interval"),
    mpatches.Patch(facecolor=C_FILL_50, label="50% Credible interval"),
    Line2D([0], [0], color=C_LINE, linewidth=2.5, marker="o", markersize=5,
           label=r"Predicted $\tau_i^{sim}$ (median)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_OBS, markersize=5,
           label=r"Observed $\tau_i^{obs}$"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_LINE, markersize=5,
           label="Seed (initial commune)"),
]
axes[0].legend(handles=legend_elements, loc="upper left", fontsize=10, frameon=False, ncol=2)
fig.tight_layout(pad=0.5)
fig.savefig(figure_path, bbox_inches="tight", pad_inches=0.01, dpi=300, facecolor="white")
plt.show()
print(f"Saved: {figure_path}")
